# Homework 5 answers

Run this notebook from `cohorts/2026/homework/05-deployment` (where `pipeline.bin`, `uv.lock`, `Dockerfile` and `predict.py` live). Run the cells top to bottom; the last cell prints a summary to copy into the form.

In [ ]:
import subprocess, json, pickle, re, time, hashlib, tomllib, urllib.request
from pathlib import Path

answers = {}

LEAD1 = {"lead_source": "paid_ads", "industry": "technology", "employment_status": "employed",
         "location": "north_america", "number_of_courses_viewed": 2, "annual_income": 79276.0,
         "interaction_count": 4, "lead_score": 0.41}
LEAD2 = {"lead_source": "organic_search", "industry": "technology", "employment_status": "employed",
         "location": "europe", "number_of_courses_viewed": 4, "annual_income": 80304.0,
         "interaction_count": 7, "lead_score": 0.74}

def post(url, payload):
    req = urllib.request.Request(url, data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.load(r)

def get(url):
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.load(r)

def wait_for(url, tries=40):
    for _ in range(tries):
        try:
            return get(url)
        except Exception:
            time.sleep(1)
    raise RuntimeError(f"{url} never came up")

## Q1: uv version

In [ ]:
out = subprocess.run(["uv", "--version"], capture_output=True, text=True).stdout.strip()
answers["q1"] = out
print(out)

## Q2: Scikit-Learn version in the lockfile

In [ ]:
lock = tomllib.loads(Path("uv.lock").read_text())
sk = [p["version"] for p in lock["package"] if p["name"] == "scikit-learn"]
answers["q2"] = sk[0] if sk else None
print(sk)

## Q3: probability from the loaded model

In [ ]:
expected = "1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1"
actual = hashlib.sha256(Path("pipeline.bin").read_bytes()).hexdigest()
print("checksum OK" if actual == expected else f"CHECKSUM MISMATCH: {actual}")

with open("pipeline.bin", "rb") as f:
    model = pickle.load(f)

p = model.predict_proba([LEAD1])[0, 1]
answers["q3"] = round(float(p), 3)
print(answers["q3"])

## Q4: probability from the API

In [ ]:
server = subprocess.Popen(["uv", "run", "uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    print(wait_for("http://localhost:9696/health"))
    resp = post("http://localhost:9696/predict", LEAD2)
    print(resp)
    answers["q4"] = round(float(resp["conversion_probability"]), 3)
finally:
    server.terminate()
    server.wait()
print(answers["q4"])

## Q5: Python base image

In [ ]:
froms = re.findall(r"^FROM\s+(\S+)", Path("Dockerfile").read_text(), flags=re.M)
print(froms)
answers["q5"] = next((f for f in froms if f.startswith("python:")), froms[0] if froms else None)
print(answers["q5"])

## Q6: probability from the container

Requires Docker running. Stop anything else using port 9696 first.

In [ ]:
subprocess.run(["docker", "build", "-t", "zoomcamp-model:2026-hw5", "."], check=True)
container = subprocess.Popen(["docker", "run", "--rm", "-p", "9696:9696", "zoomcamp-model:2026-hw5"],
                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
try:
    health = wait_for("http://localhost:9696/health")
    print("health:", health)
    resp = post("http://localhost:9696/predict", LEAD2)
    print(resp)
    answers["q6"] = round(float(resp["conversion_probability"]), 3)
finally:
    subprocess.run(["docker", "ps", "-q", "--filter", "ancestor=zoomcamp-model:2026-hw5"],
                   capture_output=True)
    ids = subprocess.run(["docker", "ps", "-q", "--filter", "ancestor=zoomcamp-model:2026-hw5"],
                         capture_output=True, text=True).stdout.split()
    for i in ids:
        subprocess.run(["docker", "stop", i])
    container.wait()
print(answers["q6"])

## Summary: copy these into the form

In [ ]:
for k in ["q1", "q2", "q3", "q4", "q5", "q6"]:
    print(k, "->", answers.get(k))

if "q4" in answers and "q6" in answers:
    print("Q4 vs Q6 within 0.005:", abs(answers["q4"] - answers["q6"]) <= 0.005)